In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
from tqdm import tqdm
import matplotlib.pyplot as plt

# === Hiperparâmetros ===
STATE_SIZE = 1
ACTION_SIZE = 1
GAMMA = 0.99
ACTOR_LR = 1e-4
CRITIC_LR = 1e-3
BATCH_SIZE = 64
MEMORY_SIZE = 100000
TAU = 0.005
MAX_STEPS = 200
MAX_EPISODES = 1000
ACTION_BOUND = 2.0
ACTION_EFFECT = 80
NOISE_STD = 0.05
rewards_history = []
erro_final_por_ep = []

# Temperatura de entropia (SAC)
ALPHA = 0.2
LOG_STD_MIN = -20
LOG_STD_MAX = 2

# === Replay Buffer ===
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))
    def __len__(self):
        return len(self.buffer)

# === Redes do SAC ===
class Actor(nn.Module):
    def __init__(self, state_size, action_size):
        super(Actor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        self.mean = nn.Linear(64, action_size)
        self.log_std = nn.Linear(64, action_size)

    def forward(self, x):
        h = self.net(x)
        mean = self.mean(h)
        log_std = self.log_std(h)
        log_std = torch.clamp(log_std, LOG_STD_MIN, LOG_STD_MAX)
        std = torch.exp(log_std)
        return mean, std, log_std

    def sample(self, x):
        mean, std, log_std = self.forward(x)
        # Reparametrização: ação não squashed
        normal = torch.distributions.Normal(mean, std)
        z = normal.rsample()
        # Squash com tanh e escala pela ACTION_BOUND
        action = torch.tanh(z) * ACTION_BOUND
        # Log-prob com correção do tanh
        log_prob = normal.log_prob(z) - torch.log(1 - torch.tanh(z).pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=1, keepdim=True)
        return action, log_prob, mean

    def act_deterministic(self, x):
        mean, _, _ = self.forward(x)
        action = torch.tanh(mean) * ACTION_BOUND
        return action

class CriticQ(nn.Module):
    def __init__(self, state_size, action_size):
        super(CriticQ, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_size + action_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        return self.fc(x)

# === Inicialização ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

actor = Actor(STATE_SIZE, ACTION_SIZE).to(device)

critic1 = CriticQ(STATE_SIZE, ACTION_SIZE).to(device)
critic2 = CriticQ(STATE_SIZE, ACTION_SIZE).to(device)
target_critic1 = CriticQ(STATE_SIZE, ACTION_SIZE).to(device)
target_critic2 = CriticQ(STATE_SIZE, ACTION_SIZE).to(device)

target_critic1.load_state_dict(critic1.state_dict())
target_critic2.load_state_dict(critic2.state_dict())

actor_optimizer = optim.Adam(actor.parameters(), lr=ACTOR_LR)
critic1_optimizer = optim.Adam(critic1.parameters(), lr=CRITIC_LR)
critic2_optimizer = optim.Adam(critic2.parameters(), lr=CRITIC_LR)

memory = ReplayBuffer(MEMORY_SIZE)
losses = []

# === Funções auxiliares ===
def select_action(state, noise=NOISE_STD):
    state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
    # SAC é estocástico; para ação de interação, use amostra da política
    action, _, _ = actor.sample(state_t)
    return action.cpu().data.numpy().flatten()

def soft_update(target, source, tau):
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)

def calculate_reward(erro_lateral):
    normalized_error = abs(erro_lateral) / 100
    reward = 1.0 - normalized_error**2
    return reward

# === Função de treinamento (SAC) ===
def train_step():
    if len(memory) < BATCH_SIZE:
        return None
    states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)

    states = torch.FloatTensor(states).to(device)
    actions = torch.FloatTensor(actions).to(device)
    rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
    next_states = torch.FloatTensor(next_states).to(device)
    dones = torch.FloatTensor(dones).unsqueeze(1).to(device)

    with torch.no_grad():
        # Próxima ação pela política atual
        next_actions, next_logp, _ = actor.sample(next_states)
        # Q-alvo mínimo
        q1_next = target_critic1(next_states, next_actions)
        q2_next = target_critic2(next_states, next_actions)
        q_next_min = torch.min(q1_next, q2_next)
        # Alvo Bellman com termo de entropia
        y = rewards + GAMMA * (1 - dones) * (q_next_min - ALPHA * next_logp)

    # Atualização dos críticos (MSE para Q1 e Q2)
    q1 = critic1(states, actions)
    q2 = critic2(states, actions)
    critic1_loss = nn.MSELoss()(q1, y)
    critic2_loss = nn.MSELoss()(q2, y)

    critic1_optimizer.zero_grad()
    critic1_loss.backward()
    critic1_optimizer.step()

    critic2_optimizer.zero_grad()
    critic2_loss.backward()
    critic2_optimizer.step()

    # Atualização do ator (maximiza Q - alpha * log_prob)
    a_pi, logp_pi, _ = actor.sample(states)
    q1_pi = critic1(states, a_pi)
    q2_pi = critic2(states, a_pi)
    q_pi_min = torch.min(q1_pi, q2_pi)
    actor_loss = (ALPHA * logp_pi - q_pi_min).mean()

    actor_optimizer.zero_grad()
    actor_loss.backward()
    actor_optimizer.step()

    # Atualização suave dos críticos-alvo
    soft_update(target_critic1, critic1, TAU)
    soft_update(target_critic2, critic2, TAU)

    return (critic1_loss.item() + critic2_loss.item()) / 2.0, actor_loss.item()

# === Ambiente de simulação ===
def simulate_step(state, action):
    erro_lateral = state[0]
    # ação é velocidade angular -> efeito sobre erro lateral
    erro_lateral += float(action) * ACTION_EFFECT
    # ruído externo (simula incerteza física)
    erro_lateral += random.uniform(-5, 5)
    erro_lateral = np.clip(erro_lateral, -100, 100)
    return np.array([erro_lateral])

# === Loop de interação ===
for episode in tqdm(range(MAX_EPISODES), desc="Treinando episódios"):
    state = np.array([random.uniform(-100, 100)])
    total_reward = 0

    for step in range(MAX_STEPS):
        action = select_action(state)
        next_state = simulate_step(state, action)
        erro_lateral = next_state[0]

        reward = calculate_reward(erro_lateral)
        done = abs(erro_lateral) > 100

        memory.push(state, action, reward, next_state, done)

        losses_val = train_step()
        if losses_val is not None:
            losses.append(losses_val)

        state = next_state
        total_reward += reward

        if done:
            break

    rewards_history.append(total_reward)
    erro_final_por_ep.append(state[0])
    print(f"[Ep {episode}] Reward: {total_reward:.2f} | Erro final: {state[0]:.2f}")

# === Salvar os modelos treinados ===
torch.save(actor.state_dict(), "sac_actor.pth")
torch.save(critic1.state_dict(), "sac_critic1.pth")
torch.save(critic2.state_dict(), "sac_critic2.pth")
print("✅ Modelos salvos (sac_actor.pth, sac_critic1.pth, sac_critic2.pth)")

# === Gráfico de recompensas ===
plt.figure(figsize=(10, 5))
plt.plot(rewards_history, label="Recompensa por episódio")
plt.xlabel("Episódio")
plt.ylabel("Recompensa total")
plt.title("Evolução da recompensa (SAC)")
plt.grid()
plt.legend()
plt.show()

# === Gráfico de erro final ===
plt.figure(figsize=(10, 5))
plt.plot(erro_final_por_ep, label="Erro lateral final")
plt.xlabel("Episódio")
plt.ylabel("Erro final")
plt.title("Convergência do erro lateral ao longo dos episódios")
plt.grid()
plt.legend()
plt.show()

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd

# === Hiperparâmetros ===
STATE_SIZE = 1
ACTION_SIZE = 1
ACTION_LIMIT = 2.0
ERRO_MAX = 300.0  # usado para normalização e visualização
LOG_STD_MIN = -20
LOG_STD_MAX = 2

# === Rede Actor (SAC) ===
class ActorSAC(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=64):
        super(ActorSAC, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU()
        )
        self.mean = nn.Linear(hidden_size, action_size)
        self.log_std = nn.Linear(hidden_size, action_size)

    def forward(self, x):
        h = self.net(x)
        mean = self.mean(h)
        log_std = self.log_std(h)
        log_std = torch.clamp(log_std, LOG_STD_MIN, LOG_STD_MAX)
        std = torch.exp(log_std)
        return mean, std

    def sample(self, x):
        mean, std = self.forward(x)
        normal = torch.distributions.Normal(mean, std)
        z = normal.sample()
        action = torch.tanh(z) * ACTION_LIMIT
        return action

    def act_deterministic(self, x):
        mean, _ = self.forward(x)
        action = torch.tanh(mean) * ACTION_LIMIT
        return action

# === Inicialização ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
actor = ActorSAC(STATE_SIZE, ACTION_SIZE).to(device)

# Carregar modelo salvo (treinado com SAC)
actor.load_state_dict(torch.load("sac_actor.pth", map_location=device))
actor.eval()
print("✅ Modelo SAC carregado!")

# === Função para escolher ação ===
def select_action(state):
    state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
    with torch.no_grad():
        action = actor.act_deterministic(state_t).cpu().numpy().flatten()
    return action[0]

# === Geração de dados ===
test_states = np.linspace(-ERRO_MAX, ERRO_MAX, 200).reshape(-1, 1)
action_list = [select_action(s) for s in test_states]

# === Exportar CSV ===
pd.DataFrame({
    "Erro_Lateral": test_states.flatten(),
    "Velocidade_Angular": action_list
}).to_csv("velocidade_predita_angular.csv", index=False)
print("📁 CSV salvo como 'velocidade_predita_angular.csv'")

# === Plot ===
plt.figure(figsize=(10, 5))
plt.plot(test_states, action_list, color='blue', label="Velocidade Angular (SAC actor)")
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.xlabel("Erro Lateral")
plt.ylabel(f"Velocidade Angular (±{ACTION_LIMIT})")
plt.title("Velocidade Angular vs Erro Lateral (SAC actor)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("velocidade_vs_erro_angular.png")
plt.show()
print("🖼️ Gráfico salvo como 'velocidade_vs_erro_angular.png'")